# Week 2 - Curve Fitting, Noisy Differentiation & Sparsity

Biology hands us measurements, and we want *parameters*: the potency of a drug,
the rate at which a signal is changing, the short list of features that actually
carry the diagnosis. Each of those is an inverse problem -- we observe a noisy
output and must reason backward to the compact quantity that produced it. This
week develops three of the workhorse tools for that job. First, **nonlinear
curve fitting**: we fit a dose-response model and, crucially, report not just a
point estimate of the potency but its *uncertainty*. Second, **regularized
differentiation**: taking a derivative of noisy data is the textbook example of
an ill-posed problem, where the naive answer is dominated by noise, and we show
how a little regularization rescues it. Third, **sparse selection**: from thirty
correlated tumor measurements we ask the Lasso to keep only the handful that
matter, and then we check whether that handful is *stable*.

The connective tissue across all three is the same honesty discipline as the
rest of the course. A fit without an error bar is a guess dressed up as a
measurement; a derivative without regularization is noise dressed up as signal;
a feature list without a stability check is an accident dressed up as biology.
Every method below is first pointed at a fixture whose true answer we already
know, so we can quantify how well it recovers ground truth before we trust it on
real data.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed. --
Chapter 4 (least-squares curve fitting, polynomial and spline interpolation, and
sparsity for curve fitting) and Chapter 3, section 4 (the numerical differentiation of
noisy data). Read those for the derivations and the finite-difference stencils;
everything below is explained in our own terms and run against our own fixtures.

**Learning goals.**

- Fit a Hill dose-response curve by nonlinear least squares and read the EC50
  and Hill coefficient *with* their standard errors.
- Judge a fit by its residual structure and goodness-of-fit, not by eye alone.
- Understand why differentiating noisy data is ill-posed, and use Tikhonov
  regularization to beat a naive finite difference against a known derivative.
- Select a sparse biomarker panel with the Lasso and test its stability by
  resampling, distinguishing robust features from lucky ones.
- Close every analysis with an explicit claim-and-limitations statement.

## Setup

We seed all random number generators and apply the course plotting style so the
figures below are deterministic and reproducible from a cold kernel.

```{admonition} Which paradigm?
:class: note
**Both / hybrid.** This week sets the two paradigms literally side by side. The Hill (four-parameter-logistic) dose-response fit is deductive and model-driven: its sigmoid comes from receptor-occupancy pharmacology, so you commit to a mechanism up front, and the parameters you recover -- EC50 as potency, the Hill slope as cooperativity -- carry meaning and let you reason beyond the doses you actually measured. That is why the same `fit_hill` estimator, validated on a synthetic curve with a known EC50, transfers unchanged to a real GDSC cancer drug-response series. The Lasso biomarker panel on the thirty Wisconsin (WDBC) tumor features is the inductive opposite: no mechanism links a nucleus's radius or concavity to malignancy, so you let the data pick the sparse subset that predicts and use resampling to see which features are stable. Reach for the mechanistic model when the functional form is known and you want interpretable parameters or extrapolation; reach for the flexible, regularized fit when the form is unknown and you only need to learn which measurements carry the signal.
```


In [ ]:
# Colab setup: install the ddm4bio course library.
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git"
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()

print(f"ddm4bio version: {ddm4bio.__version__}")

## 1. Fitting a dose-response curve with known ground truth

A dose-response experiment measures how a readout (cell viability, receptor
occupancy, fluorescence) changes as we sweep the concentration of a compound
across several orders of magnitude. The **Hill equation** is the standard
sigmoid model for that curve:

$$ y(x) = \text{bottom} + (\text{top} - \text{bottom})\,\frac{x^{n}}{\text{EC50}^{n} + x^{n}}. $$

Two of its four parameters are the ones pharmacologists actually report. The
**EC50** is the concentration producing a half-maximal response -- the standard
summary of potency. The **Hill coefficient** $n$ controls the steepness of the
transition and is often read as a measure of cooperativity. Fitting the curve is
easy; the discipline is reporting how *precisely* those two numbers are pinned
down by noisy data.

To test the fitter honestly we build a synthetic experiment whose true
parameters we choose ourselves. We lay out ten concentrations spaced
logarithmically from 0.01 to 100 (a typical four-log dose range), take three
replicate measurements at each concentration, and add Gaussian measurement noise.
An honest fitter should recover the true EC50 and Hill coefficient to within the
uncertainty it reports.

In [ ]:
from ddm4bio.methods.fitting import hill, fit_hill

rng = np.random.default_rng(1)

# Ground-truth Hill parameters we will try to recover.
true = {"bottom": 5.0, "top": 95.0, "ec50": 1.2, "n": 1.8}

concentrations = np.geomspace(0.01, 100.0, 10)  # 10 doses over 4 logs
n_replicates = 3
x = np.repeat(concentrations, n_replicates)       # 3 replicates per dose

y_clean = hill(x, true["bottom"], true["top"], true["ec50"], true["n"])
y = y_clean + rng.normal(0.0, 2.5, size=x.size)   # additive measurement noise

print(f"Design: {concentrations.size} concentrations x {n_replicates} replicates "
      f"= {x.size} measurements")
print(f"True EC50 = {true['ec50']}, true Hill coefficient = {true['n']}")

Now fit the Hill model by nonlinear least squares. `fit_hill` wraps
`scipy.optimize.curve_fit`, which returns both the best-fit parameters and their
covariance matrix; the square roots of its diagonal are the 1-sigma standard
errors on each parameter. We print the fitted EC50 and Hill coefficient next to
their true values, each with a standard error.

In [ ]:
fit = fit_hill(x, y, seed=0)

ec50_hat, ec50_se = fit["ec50"], fit["std_errors"][2]
n_hat, n_se = fit["hill_coeff"], fit["std_errors"][3]

print(f"fit converged: {fit['success']}")
print(f"EC50:            {ec50_hat:6.3f}  +/- {ec50_se:.3f}   (true {true['ec50']})")
print(f"Hill coeff n:    {n_hat:6.3f}  +/- {n_se:.3f}   (true {true['n']})")
print(f"bottom:          {fit['bottom']:6.3f}            (true {true['bottom']})")
print(f"top:             {fit['top']:6.3f}            (true {true['top']})")

A point estimate near the truth is reassuring, but the honest question is
whether the *true* value sits inside the confidence interval the fit reports. We
express the gap between fitted and true as a z-score -- the number of standard
errors between them. A z-score comfortably inside +/-2 means the true value is
consistent with the fit at roughly the 95% level; a large z-score would warn
that the fit is biased or the error bar is too optimistic.

In [ ]:
z_ec50 = (ec50_hat - true["ec50"]) / ec50_se
z_n = (n_hat - true["n"]) / n_se

print(f"EC50 recovered within {z_ec50:+.2f} standard errors of truth")
print(f"Hill coefficient recovered within {z_n:+.2f} standard errors of truth")

The figure below overlays the fitted curve on the replicate measurements. Dose
is plotted on a log axis (the natural scale for a four-log concentration sweep),
the true curve is drawn as a dashed reference, and a vertical marker shows the
fitted EC50 with its uncertainty shaded.

In [ ]:
import matplotlib.pyplot as plt

xx = np.geomspace(concentrations.min(), concentrations.max(), 300)
y_fit = hill(xx, *fit["params"])
y_true_curve = hill(xx, true["bottom"], true["top"], true["ec50"], true["n"])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, s=28, alpha=0.7, label="measurements", zorder=3)
ax.plot(xx, y_fit, linewidth=2, label="fitted Hill curve")
ax.plot(xx, y_true_curve, linestyle="--", linewidth=1.5, label="true curve")
ax.axvspan(ec50_hat - ec50_se, ec50_hat + ec50_se, alpha=0.15,
           color="0.4", label="EC50 +/- 1 s.e.")
ax.axvline(ec50_hat, color="0.4", linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("Concentration (arb. units, log scale)")
ax.set_ylabel("Response (% of maximum)")
ax.set_title("Hill dose-response fit vs. ground truth")
ax.legend(loc="upper left", fontsize=9)
fig;

**QC: goodness of fit and residual structure.** A good fit is not just close on
average -- its residuals should look like structureless noise. If the residuals
curved systematically with concentration, that would signal a *model*
mismatch (wrong functional form) rather than mere measurement scatter. We report
the coefficient of determination $R^2$ and plot the residuals against dose,
looking for the flat, patternless band that signals a well-specified model.

In [ ]:
y_hat = hill(x, *fit["params"])
residuals = y - y_hat
ss_res = float(np.sum(residuals**2))
ss_tot = float(np.sum((y - y.mean())**2))
r_squared = 1.0 - ss_res / ss_tot
rmse = float(np.sqrt(np.mean(residuals**2)))

print(f"R^2  = {r_squared:.4f}")
print(f"RMSE = {rmse:.3f} response units "
      f"(noise was drawn with sigma = 2.5)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.axhline(0.0, color="0.5", linewidth=1)
ax.scatter(x, residuals, s=28, alpha=0.7)
ax.set_xscale("log")
ax.set_xlabel("Concentration (log scale)")
ax.set_ylabel("Residual")
ax.set_title("Residuals vs. dose (should be a structureless band)")
fig;

**QC note.** The RMSE lands close to the 2.5-unit noise we injected, and the
residuals scatter around zero with no visible trend across the dose range --
exactly the signature of a correctly specified model whose only error is the
measurement noise we put in. The fit has not "used up" the data explaining
structure that isn't there.

### Fitting the same model to a real GDSC dose-response series

The synthetic curve above let us *verify* the fitter against a truth we chose.
Now we point the identical machinery at a real pharmacology resource: the
Genomics of Drug Sensitivity in Cancer (GDSC) project, which screened hundreds
of compounds across a large panel of cancer cell lines. We pull it through the
course data layer, which fetches and caches the real GDSC dose-response table
when a network is available and otherwise hands back a clearly-labelled
synthetic stand-in with the *same* long-format table shape. The provenance line
tells the reader which one they got, and nothing downstream depends on the
answer.

In [ ]:
from ddm4bio.datasets import get_dataset

ds = get_dataset("gdsc", seed=0)  # real GDSC download (cached), else labelled fallback
print(f"gdsc source     : {ds.source}")
print(f"gdsc provenance : {ds.provenance}")

table = ds.payload  # a pandas DataFrame of dose-response measurements
print(f"table shape     : {table.shape[0]} rows x {table.shape[1]} columns")
print(f"columns         : {list(table.columns)}")

Public tables come in two very different shapes, and a robust reader must
tolerate both. Some record one row per measured *concentration* (a raw
dose-response series we can fit directly); others store one row per *fitted
curve*, keeping only summary statistics such as a reported half-maximal
concentration. We write one defensive extractor that returns a single
dose-response series either way: it prefers an explicit response column measured
across concentrations, and otherwise reconstructs the fitted sigmoid for one
real drug/cell-line pair from its reported (log) IC50 over the assay's
concentration range.

In [ ]:
def _find_col(frame, keywords):
    """First column whose lowercased name contains any of the keywords."""
    for col in frame.columns:
        if any(k in str(col).lower() for k in keywords):
            return col
    return None

_ID_NAMES = {"cell_line", "cell_line_name", "drug", "drug_name", "drug_id", "cosmic_id"}


def extract_series(frame):
    """Return (x, y, label, dose_name, resp_name) for one GDSC series.

    Handles a long per-concentration table and a fitted-summary table alike.
    """
    resp_col = _find_col(frame, ("viability", "response", "inhib", "signal", "activity"))
    dose_col = _find_col(frame, ("concentration",))
    id_cols = [c for c in frame.columns if str(c).lower() in _ID_NAMES]

    if resp_col is not None and dose_col is not None:  # raw per-concentration table
        if id_cols:
            grouped = frame.groupby(id_cols)
            key = grouped.size().idxmax()  # series with the most concentrations
            sub = grouped.get_group(key).sort_values(dose_col)
            label = " / ".join(str(v) for v in (key if isinstance(key, tuple) else (key,)))
        else:
            sub, label = frame.sort_values(dose_col), "all rows"
        return (sub[dose_col].to_numpy(float), sub[resp_col].to_numpy(float),
                label, str(dose_col), str(resp_col))

    # Fitted-summary table: rebuild the sigmoid from a reported (log) IC50.
    ic50_col = _find_col(frame, ("ln_ic50", "ic50", "ic_50"))
    lo_col = _find_col(frame, ("min_conc", "min_dose", "conc_min"))
    hi_col = _find_col(frame, ("max_conc", "max_dose", "conc_max"))
    if ic50_col is None:
        raise ValueError("GDSC table lacks both raw responses and an IC50 column")
    row = frame.sort_values(ic50_col).iloc[len(frame) // 2]  # median-potency curve
    ic50 = float(np.exp(row[ic50_col]) if "ln" in str(ic50_col).lower() else row[ic50_col])
    lo = float(row[lo_col]) if lo_col is not None else ic50 / 100.0
    hi = float(row[hi_col]) if hi_col is not None else ic50 * 100.0
    x = np.geomspace(max(lo, 1e-6), max(hi, lo * 10.0), 12)
    y = 1.0 / (1.0 + (x / ic50))  # GDSC-style fitted viability sigmoid
    id_bits = [str(row[c]) for c in id_cols] or ["reconstructed"]
    return x, y, " / ".join(id_bits) + " (from fitted IC50)", "concentration (uM)", "fitted viability"


x_real, y_real, label, dose_col, resp_col = extract_series(table)
print(f"series: {label}  ({x_real.size} concentrations)")
print(f"dose column = {dose_col!r}, response column = {resp_col!r}")

Now the same `fit_hill` call as in the ground-truth section, on real (or
realistic fallback) numbers. A viability readout *falls* as dose rises, so the
fitted sigmoid is a decreasing one; the sign of the recovered Hill coefficient
just encodes that direction, while its magnitude is the steepness of the
transition. The EC50 marks the half-maximal concentration, reported with its
standard error.

In [ ]:
gdsc_fit = fit_hill(x_real, y_real, seed=0)

print(f"fit converged     : {gdsc_fit['success']}")
print(f"EC50              : {gdsc_fit['ec50']:.4g} "
      f"+/- {gdsc_fit['std_errors'][2]:.2g}")
print(f"Hill coefficient  : {gdsc_fit['hill_coeff']:+.3g} "
      f"(|value| = steepness; sign = direction)")

In [ ]:
lo = float(x_real[x_real > 0].min())
xx = np.geomspace(lo, float(x_real.max()), 300)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x_real, y_real, s=32, alpha=0.8, zorder=3, label="GDSC measurements")
ax.plot(xx, hill(xx, *gdsc_fit["params"]), linewidth=2, label="fitted Hill curve")
ax.axvline(gdsc_fit["ec50"], color="0.4", linewidth=1, label="fitted EC50")
ax.set_xscale("log")
ax.set_xlabel(f"{dose_col} (log scale)")
ax.set_ylabel(str(resp_col))
ax.set_title(f"Hill fit to a real GDSC series ({label})")
ax.legend(loc="best", fontsize=9)
fig;

The point of this pass is the *transfer*: the estimator we validated on a curve
with a known answer behaves identically on a public dataset whose answer we did
not know in advance. The ground-truth section earned the trust; this section
spends it.

## 2. Differentiating a noisy signal

Many biological questions are really questions about a *rate*: the velocity of a
growth curve, the acceleration of a tumor volume, the slope of a metabolic
trace. That means differentiating measured data -- and differentiation is the
canonical ill-posed problem. The derivative operator amplifies high frequencies,
and measurement noise is almost all high frequency, so a naive finite difference
turns a barely-visible wiggle in the data into a wild oscillation in the
estimated derivative.

To see this cleanly we again start from ground truth: a pure sine wave, whose
derivative we know analytically ($\frac{d}{dt}\sin(2\pi f t) = 2\pi f\cos(2\pi f t)$).
We sample it on a uniform grid, corrupt it with a small amount of noise, and
then compare two derivative estimates against the exact answer -- a plain
finite difference (`numpy.gradient`) versus the Tikhonov-regularized derivative
from `regularized_derivative`, which finds the smooth function whose integral
best matches the data.

In [ ]:
from ddm4bio.methods.fitting import regularized_derivative

rng = np.random.default_rng(0)

n = 200
t = np.linspace(0.0, 1.0, n, endpoint=False)
dt = t[1] - t[0]
freq = 2.0

clean = np.sin(2.0 * np.pi * freq * t)
true_derivative = 2.0 * np.pi * freq * np.cos(2.0 * np.pi * freq * t)  # analytic
noisy = clean + rng.normal(0.0, 0.05, size=n)  # only 5% noise on the signal

finite_diff = np.gradient(noisy, dt)
reg_deriv = regularized_derivative(noisy, dt, lam=1e-2)

def rel_error(estimate):
    """Relative L2 error of a derivative estimate against the analytic truth."""
    return float(np.linalg.norm(estimate - true_derivative)
                 / np.linalg.norm(true_derivative))

print(f"Signal noise level: 5% of amplitude")
print(f"Finite-difference   relative error: {rel_error(finite_diff):.3f}")
print(f"Regularized         relative error: {rel_error(reg_deriv):.3f}")

The numbers tell the whole story: a 5% wiggle in the *signal* becomes a roughly
70% error in the naive derivative, while regularization keeps the error near
10%. The plot makes the mechanism visible. The finite difference (top) is buried
in noise; the regularized estimate (bottom) tracks the true cosine closely.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

axes[0].plot(t, true_derivative, linewidth=2, label="true derivative")
axes[0].plot(t, finite_diff, linewidth=1, alpha=0.8, label="finite difference")
axes[0].set_ylabel("dy/dt")
axes[0].set_title("Naive finite difference amplifies the noise")
axes[0].legend(loc="upper right", fontsize=9)

axes[1].plot(t, true_derivative, linewidth=2, label="true derivative")
axes[1].plot(t, reg_deriv, linewidth=1.5, linestyle="--",
             label="regularized derivative")
axes[1].set_xlabel("t")
axes[1].set_ylabel("dy/dt")
axes[1].set_title("Tikhonov regularization recovers the derivative")
axes[1].legend(loc="upper right", fontsize=9)
fig;

**Choosing the regularization strength.** The smoothing parameter `lam` is a
dial between two failure modes: too little and noise leaks through, too much and
genuine features get flattened. Sweeping it shows a broad basin of good values
rather than a single knife-edge -- reassuring, because it means we do not have
to tune the parameter to unrealistic precision.

In [ ]:
lambdas = [1e-4, 1e-3, 1e-2, 3e-2, 1e-1]
errors = [rel_error(regularized_derivative(noisy, dt, lam=lam)) for lam in lambdas]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lambdas, errors, marker="o", linewidth=1.5, label="regularized")
ax.axhline(rel_error(finite_diff), color="0.5", linestyle="--",
           label="finite difference")
ax.set_xscale("log")
ax.set_xlabel("Regularization strength lambda")
ax.set_ylabel("Relative L2 error vs. truth")
ax.set_title("A broad basin of good regularization strengths")
ax.legend(loc="center right", fontsize=9)
fig;

## 3. Sparse biomarker selection on a real dataset

We now leave synthetic fixtures for a real biomedical dataset, pulled through
the course data layer: the Wisconsin Diagnostic Breast Cancer measurements. Each
of 569 tumors is described by 30 features computed from a digitized image of a
fine-needle aspirate -- radius, texture, concavity and so on, each summarized
as a mean, a standard error, and a "worst" (largest) value -- and labeled
benign or malignant. Many of these 30 features are strongly correlated with one
another. A clinician does not want thirty numbers; they want the *few* that
carry the diagnostic signal.

The **Lasso** is built for exactly this. By penalizing the sum of absolute
coefficients it drives most of them to *exactly* zero, keeping only a compact
subset. We standardize the features first (so the penalty treats them on equal
footing) and fit the Lasso at a penalty strong enough to force a small panel.

In [ ]:
from sklearn.preprocessing import StandardScaler

from ddm4bio.datasets import get_dataset
from ddm4bio.methods.fitting import lasso_select

bc = get_dataset("breast_wisconsin")  # bundled WDBC (real); labelled fallback offline
print(f"breast_wisconsin source     : {bc.source}")
print(f"breast_wisconsin provenance : {bc.provenance}")

feature_names = np.asarray(bc.payload["feature_names"])
X = StandardScaler().fit_transform(np.asarray(bc.payload["X"], dtype=float))
y = np.asarray(bc.payload["y"], dtype=float)    # 0 = malignant, 1 = benign

print(f"Design matrix: {X.shape[0]} tumors x {X.shape[1]} features")

result = lasso_select(X, y, alpha=0.05, seed=0)
selected = result["selected"]

print(f"\nLasso kept {selected.size} of {X.shape[1]} features "
      f"(alpha = {result['alpha']}):")
for idx in selected:
    print(f"  {feature_names[idx]:24s}  coef = {result['coefficients'][idx]:+.3f}")

**QC: stability of the selected features.** A single Lasso fit is one draw from
one sample. If we had collected a slightly different cohort, would we get the
same panel? The honest way to find out is to *resample*: we draw many bootstrap
replicates of the patients, re-run the selection on each, and count how often
each feature is chosen. Features selected in nearly every replicate are robust;
features that flicker in and out are artifacts of this particular sample and
should not be trusted as biomarkers.

In [ ]:
n_boot = 50
n_patients = X.shape[0]
selection_counts = np.zeros(X.shape[1])

boot_rng = np.random.default_rng(0)
for _ in range(n_boot):
    idx = boot_rng.integers(0, n_patients, size=n_patients)  # sample with replacement
    boot_result = lasso_select(X[idx], y[idx], alpha=0.05, seed=0)
    selection_counts[boot_result["selected"]] += 1

selection_freq = selection_counts / n_boot

# Rank features by how often they survived resampling.
order = np.argsort(-selection_freq)
print(f"Selection frequency across {n_boot} bootstrap resamples:")
print("(* marks features also chosen on the full sample)\n")
for idx in order[:10]:
    star = "*" if idx in selected else " "
    print(f"  {star} {feature_names[idx]:24s}  {selection_freq[idx]:.2f}")

The bar chart below makes the split obvious. A few features -- the "worst" (i.e.
largest-valued) concave-points, radius, and texture measurements -- are selected
in essentially every resample, marking them as the stable diagnostic core.
Others hover near a coin flip and would be irresponsible to report as
established biomarkers on this evidence.

In [ ]:
top = order[:10]
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#0072B2" if idx in selected else "#D55E00" for idx in top]
ax.barh(range(len(top)), selection_freq[top], color=colors)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([feature_names[idx] for idx in top])
ax.invert_yaxis()  # most frequent at the top
ax.axvline(0.5, color="0.5", linestyle="--", linewidth=1, label="coin flip")
ax.set_xlabel("Fraction of bootstrap resamples that selected the feature")
ax.set_title("Feature-selection stability under resampling")
ax.legend(loc="lower right", fontsize=9)
fig;

In [ ]:
stable = [feature_names[idx] for idx in order if selection_freq[idx] >= 0.8]
print(f"Features selected in >= 80% of resamples ({len(stable)}):")
for name in stable:
    print(f"  - {name}")

## 4. Interpretation

Every ddm4bio analysis closes with an explicit interpretation block: a single
claim, stated with the evidence that backs it, and a list of named
limitations. Here we make two claims -- one about the recovered potency, one
about which biomarkers are trustworthy -- and pin each to the evidence we
actually generated above.

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

n_stable = int(np.sum(selection_freq >= 0.8))

block = interpretation_block(
    claim=(
        f"The Hill fit recovers the true potency (EC50 = {ec50_hat:.2f} +/- "
        f"{ec50_se:.2f}, true 1.20) and cooperativity within its reported "
        f"uncertainty; regularized differentiation cuts the derivative error from "
        f"{rel_error(finite_diff):.2f} to {rel_error(reg_deriv):.2f} against the analytic "
        f"truth; and the breast-cancer biomarker panel has a stable core "
        f"of {n_stable} features (led by worst concave points, worst radius, "
        f"and worst texture) that survive resampling."
    ),
    limitations_list=[
        f"The EC50 confidence interval assumes Gaussian, homoscedastic noise; "
        f"real assays often have concentration-dependent variance that would "
        f"widen the true interval beyond the reported +/-{ec50_se:.2f}.",
        "Standard errors from curve_fit are asymptotic (linearized) and can "
        "understate uncertainty for a small design like 10 doses x 3 replicates.",
        "The Hill fit was validated on a synthetic curve with a known form; a "
        "genuinely biphasic or model-mismatched response would fit poorly "
        "despite a clean-looking single-model R^2.",
        "Lasso stability was assessed at one fixed penalty (alpha = 0.05); a "
        "different penalty changes the panel size, and the borderline features "
        "(worst smoothness, mean texture) are penalty-sensitive.",
        "Selection frequency measures reproducibility on THIS cohort, not "
        "biological causality or generalization to a new patient population.",
    ],
)
show_interpretation(block)

## Exercises

Your graded work for this week is **Problem Set 2 (PS2) -- "Letting the Data Choose the Model:
Cross-Validated Selection"**, distributed and auto-graded through GitHub Classroom. Week 2 fit
models you specified; PS2 is about *selecting* among them honestly -- letting held-out folds,
not the training fit, choose the complexity and the feature set.

- **Part A -- choose model complexity by cross-validation.** Implement `poly_cv_mse` (the mean
  out-of-sample MSE of a degree-`d` polynomial over k folds) and `select_degree` (the degree
  that minimizes it), and recover the true degree from noisy data.
- **Part B -- choose a sparse feature set by cross-validation.** Implement `lasso_cv_mse`,
  `select_alpha`, `selected_features`, and `support_scores` (precision/recall of the selected
  set), then apply them to a real biomarker panel and close with an interpretation block.

Refer to the [PS2 repository README](https://github.com/symbiont-ai/ddm4bio/tree/main/problem_sets/ps2_curvefit_sparsity) for the submission and auto-grading details.